# Bootstrap intervals for Krippendorff's alpha: which one to trust

<a href="https://colab.research.google.com/github/Sathapana/reliability-lab/blob/main/notebooks/tutorial_bootstrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

An alpha computed from 200 units is an estimate. Another 200 units, or another panel of coders, would give a different value. A **bootstrap** interval says how different, by recomputing alpha on many resampled copies of the data.

reliability-lab has no default bootstrap. You must name a `scheme`, because each scheme answers a different question. This notebook shows:

1. what the bootstrap does, by building one by hand;
2. how to check that a 95% interval really covers 95% of the time;
3. percentile and BCa intervals, and what happens with small, skewed samples;
4. why KALPHA intervals get too narrow with many coders;
5. why groups of related units must be resampled together;
6. which interval to report when the claim is about **new coders**, not just new units.

It follows the [bootstrap coverage guidance](https://github.com/Sathapana/reliability-lab/blob/main/docs/guidance/bootstrap-coverage.md), which measures all of this at full scale. Here every experiment is small enough to run in seconds, so you can change the settings and rerun. If you are new to alpha, start with the [coder reliability tutorial](https://github.com/Sathapana/reliability-lab/blob/main/notebooks/tutorial_coder_reliability.ipynb).

## Setup

In [ ]:
%pip install -q reliability-lab matplotlib

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reliability_lab import CoderSpec, RatingFrame, ReliabilityStudy

pd.set_option("display.precision", 3)

## The coders and their ratings

The simulation separates **who the coders are** from **what they rate**, so we can later draw new units for the same coders, or new coders for new units.

- `draw_coders` draws a panel: each coder's accuracy is spread around a mean accuracy by `spread`.
- `rate` has a panel rate a set of units. A coder who knows the answer gives it; otherwise they guess, in proportion to how common each answer is. 30% of units are hard, and every coder's accuracy drops by 0.25 on them.
- With `cluster_size > 1`, units come in groups (turns of one conversation, say) that share their difficulty and, usually, their answer.

In [ ]:
def draw_coders(n_coders=10, accuracy=0.8, spread=0.1, seed=None):
    """One panel of coders: the accuracy of each."""
    rng = np.random.default_rng(seed)
    return np.clip(accuracy + spread * rng.standard_normal(n_coders), 0.05, 1.0)


def rate(coder_accuracy, n_units=200, values=("A", "B"), prevalence=None, cluster_size=1, seed=None):
    """The panel's ratings of new units: a units x coders table, and each unit's group."""
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=object)
    k, m = len(values), len(coder_accuracy)
    p = np.full(k, 1 / k) if prevalence is None else np.asarray(prevalence, dtype=float)

    group = np.arange(n_units) // cluster_size
    n_groups = group[-1] + 1
    hard = (rng.random(n_groups) < 0.3)[group]
    if cluster_size > 1:
        topic = rng.choice(k, size=n_groups, p=p)[group]
        truth = np.where(rng.random(n_units) < 0.7, topic, rng.choice(k, size=n_units, p=p))
    else:
        truth = rng.choice(k, size=n_units, p=p)

    accuracy = np.clip(coder_accuracy[None, :] - 0.25 * hard[:, None], 0.0, 1.0)
    knows = rng.random((n_units, m)) < accuracy
    answer = np.where(knows, truth[:, None], rng.choice(k, size=(n_units, m), p=p))

    units = pd.Index([f"u{i + 1:05d}" for i in range(n_units)], name="unit")
    wide = pd.DataFrame(values[answer], index=units, columns=[f"coder_{j + 1:02d}" for j in range(m)])
    return wide, pd.Series([f"g{g:04d}" for g in group], index=units, name="group")


def make_study(wide, values=("A", "B"), groups=None):
    """A nominal human-panel study of a units x coders table."""
    long = wide.reset_index().melt(id_vars="unit", var_name="coder", value_name="answer")
    extra = {}
    if groups is not None:
        long["group"] = long["unit"].map(groups)
        extra["group"] = "group"
    ratings = RatingFrame.from_long(long, unit="unit", rater="coder", value="answer", **extra)
    return ReliabilityStudy(
        ratings,
        measurement="nominal",
        values=list(values),
        coders=CoderSpec(by="rater_id"),
        design="human_panel",
        missing_policies=["as_missing"],
    )

In [ ]:
panel = draw_coders(seed=7)
ratings, _ = rate(panel, seed=8)
result = make_study(ratings).alpha()
print(f"10 coders x {len(ratings)} units: alpha = {result.alpha:.3f}")
ratings.head()

## 1. The bootstrap by hand

Pretend the 200 units we have are the population. Draw 200 units from them **with replacement**: some units appear twice, some not at all. Compute alpha on that copy. Repeat many times. The spread of those alphas shows how much alpha moves from one sample of units to the next, and the middle 95% of them is a **percentile interval**.

Each unit is resampled whole, with all ten of its ratings, because ratings of one unit are not independent of each other.

In [ ]:
rng = np.random.default_rng(1)
by_hand = []
for _ in range(200):
    rows = rng.integers(0, len(ratings), size=len(ratings))
    copy = ratings.iloc[rows].reset_index(drop=True)
    copy.index = pd.Index([f"r{i}" for i in range(len(copy))], name="unit")  # a repeated unit counts twice
    by_hand.append(make_study(copy).alpha().alpha)

low, high = np.percentile(by_hand, [2.5, 97.5])
print(f"by hand, 200 resamples: 95% interval [{low:.3f}, {high:.3f}]")

`scheme="units"` does the same thing, much faster. `random_state` fixes the resamples, so the interval is reproducible; the seed is recorded in the result. Every replicate is kept in the interval's `distribution`.

In [ ]:
boot = result.bootstrap(scheme="units", n_resamples=2000, random_state=7)
print(boot.summary())

In [ ]:
interval = boot["as_missing"]
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(interval.distribution, bins=40, alpha=0.6, density=True, label="scheme='units', 2000 resamples")
ax.hist(by_hand, bins=20, alpha=0.6, density=True, label="by hand, 200 resamples")
for x in (interval.low, interval.high):
    ax.axvline(x, color="black", linestyle="--", linewidth=1)
ax.axvline(result.alpha, color="tab:red", label="alpha of the sample")
ax.set_xlabel("alpha in a bootstrap replicate")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2. Does a 95% interval cover 95% of the time?

In a simulation we can know the true alpha, so we can check an interval rather than trust it.

1. **Truth**: alpha of the same coders on 20,000 units, close to its population value.
2. **Repeat 200 times**: draw 50 fresh units, compute a units interval, and record whether it contains the truth.

A good 95% interval contains the truth in about 95% of the repetitions. This is the experiment behind the guidance's tables, which ran 2,000 repetitions with 2,000 resamples each.

In [ ]:
big, _ = rate(panel, n_units=20_000, seed=99)
true_alpha = make_study(big).alpha().alpha
print(f"true alpha of these coders: {true_alpha:.3f}")

In [ ]:
def coverage_run(n_units=50, n_reps=200, n_resamples=300, method="percentile", **rate_args):
    rows = []
    for rep in range(n_reps):
        sample, _ = rate(panel, n_units=n_units, seed=1000 + rep, **rate_args)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            iv = make_study(sample).alpha().bootstrap(
                scheme="units", method=method, n_resamples=n_resamples, random_state=rep
            )["as_missing"]
        rows.append({"point": iv.point, "low": iv.low, "high": iv.high})
    runs = pd.DataFrame(rows)
    runs["covers"] = (runs["low"] <= true_alpha) & (true_alpha <= runs["high"])
    return runs


runs = coverage_run()
print(f"coverage: {runs['covers'].mean():.1%} of {len(runs)} intervals contain the true alpha")
print(f"missed below: {(runs['high'] < true_alpha).mean():.1%}, missed above: {(runs['low'] > true_alpha).mean():.1%}")

In [ ]:
shown = runs.head(60)
fig, ax = plt.subplots(figsize=(7, 5))
colors = np.where(shown["covers"], "tab:blue", "tab:red")
ax.hlines(range(len(shown)), shown["low"], shown["high"], colors=colors)
ax.plot(shown["point"], range(len(shown)), "o", color="black", markersize=2)
ax.axvline(true_alpha, color="black", linestyle="--", label="true alpha")
ax.set_xlabel("alpha")
ax.set_ylabel("repetition")
ax.set_title("60 of the intervals; red ones miss the truth")
ax.legend()
plt.tight_layout()
plt.show()

With 200 repetitions the coverage itself has a Monte Carlo error of about 1.5 points. Here the percentile interval falls a little short, and almost all its misses fall **below** the truth: on small samples alpha is biased slightly low, and the percentile interval inherits the bias. The next section looks at a fix.

**Try it:** set `n_units=200` in `coverage_run`, and see whether the misses even out.

## 3. Percentile or BCa

The percentile interval takes the middle 95% of the replicates as they are. **BCa** (bias-corrected and accelerated) shifts and stretches that range to correct for a replicate distribution that is biased or skewed. Alpha on small samples is biased slightly low, and in the guidance's simulations BCa gave the most accurate coverage of any interval.

In [ ]:
for method in ("percentile", "bca"):
    iv = result.bootstrap(scheme="units", method=method, n_resamples=2000, random_state=7)["as_missing"]
    print(f"{method:<10} [{iv.low:.3f}, {iv.high:.3f}]  width {iv.high - iv.low:.3f}")

On 200 well-behaved units the two agree. Now rerun the coverage experiment of section 2, with 50 units, using BCa:

In [ ]:
bca_runs = coverage_run(method="bca", n_resamples=500)
for name, r in (("percentile", runs), ("bca", bca_runs)):
    print(f"{name:<10} coverage {r['covers'].mean():.1%}   missed below {(r['high'] < true_alpha).mean():.1%}"
          f"   missed above {(r['low'] > true_alpha).mean():.1%}")

BCa brings coverage closer to 95% and shifts fewer misses below the truth. It has a weakness, though: it can fail on **small samples dominated by one category**. Below, 3 coders rate 30 units, 80% of which are `A`. Some resamples contain almost only `A`, which leaves alpha undefined. The library never hides that: it counts the failed replicates (`n_failed`) and warns. BCa needs every replicate and its jackknife, so it may give no interval at all; the percentile interval is then still computed from the defined replicates.

`alpha_min` adds one more number: the share of replicates below a reliability threshold, here Krippendorff's 0.667 for tentative conclusions.

In [ ]:
small, _ = rate(draw_coders(3, seed=2), n_units=30, values=("A", "B", "C"), prevalence=[0.8, 0.1, 0.1], seed=9)
small_result = make_study(small, values=("A", "B", "C")).alpha()
print(f"alpha = {small_result.alpha:.3f}")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    rows = []
    for method in ("percentile", "bca"):
        iv = small_result.bootstrap(
            scheme="units", method=method, n_resamples=2000, alpha_min=0.667, random_state=7
        )["as_missing"]
        rows.append({"method": method, "low": iv.low, "high": iv.high, "n_failed": iv.n_failed,
                     "share below 0.667": iv.p_below_min, "undefined_reason": iv.undefined_reason})

for w in caught:
    print(f"{w.category.__name__}: {w.message}")
pd.DataFrame(rows).set_index("method")

Note how wide the interval is with 30 units: it runs from almost no reliability to good reliability. Small studies can rarely support a claim that alpha clears a threshold.

## 4. KALPHA: too narrow with many coders

`scheme="kalpha"` reproduces Krippendorff's KALPHA algorithm, which resamples **pairs of values** rather than units, and holds the expected disagreement fixed. A unit rated by `m` coders holds `m(m - 1)/2` pairs, and those pairs share the unit's difficulty. Resampling them as if they were independent understates the uncertainty, more so the more coders there are.

In [ ]:
rows = []
for n_coders in (2, 10):
    sample, _ = rate(draw_coders(n_coders, spread=0, seed=3), seed=4)
    res = make_study(sample).alpha()
    for scheme in ("units", "kalpha"):
        iv = res.bootstrap(scheme=scheme, n_resamples=2000, random_state=7)["as_missing"]
        rows.append({"coders": n_coders, "scheme": scheme, "alpha": res.alpha, "low": iv.low,
                     "high": iv.high, "width": iv.high - iv.low})
pd.DataFrame(rows).set_index(["coders", "scheme"])

With 2 coders each unit holds one pair, and the two schemes agree. With 10 coders the KALPHA interval is less than half as wide as the units interval. In the guidance's simulations it covered 70–71% with only 5 coders. Use it to reproduce a published KALPHA interval, not to make a new claim.

## 5. Related units: resample the groups

Units are often not independent: several turns of one conversation, several responses to one prompt. Here units come in groups of 5 that share their difficulty and usually their answer. Declare the group with `group=` in `RatingFrame.from_long`, and the units scheme resamples **whole groups** by default.

In [ ]:
clustered, groups = rate(panel, n_units=200, cluster_size=5, seed=5)
clustered_result = make_study(clustered, groups=groups).alpha()

rows = []
for level in ("unit", "group"):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        iv = clustered_result.bootstrap(scheme="units", level=level, n_resamples=2000, random_state=7)["as_missing"]
    rows.append({"resampled": level, "low": iv.low, "high": iv.high, "width": iv.high - iv.low})
    for w in caught:
        print(f"level={level}: {w.category.__name__}: {w.message}")
pd.DataFrame(rows).set_index("resampled")

Resampling units one by one treats 200 units as 200 independent pieces of evidence, when there are really 40 groups. The group-level interval is wider, and correct: in the guidance, unit-level resampling of clustered data covered 89–91%, group-level 92–95%.

## 6. Whose reliability? New units versus new coders

So far the coders were fixed: every interval spoke about **these coders** rating other units. Often the claim is broader: *trained annotators like these*, or *this LLM judge* when the 10 coders are 10 runs. Then the coders are a sample too, and the interval has to include how much alpha changes with the panel.

The three claims need three intervals:

| Claim | Interval | Resampled |
| --- | --- | --- |
| These coders, on new units | `scheme="units"` | units (or groups) |
| These units, with new coders like these | `result.coder_jackknife()` | coders |
| New coders like these, on new units | `scheme="two_way"` | units and coders |

To see the difference, we draw coders from a pool where they **differ a lot** (`spread=0.2`: accuracies from about 0.4 to 1).

In [ ]:
POOL = dict(accuracy=0.8, spread=0.2)

study_panel = draw_coders(10, **POOL, seed=21)
sample, _ = rate(study_panel, seed=22)
differ = make_study(sample).alpha()
print("accuracy of the study's coders:", np.round(study_panel, 2))
print(f"alpha = {differ.alpha:.3f}")

In [ ]:
intervals = {}
for scheme in ("units", "two_way", "coders"):
    intervals[scheme] = differ.bootstrap(scheme=scheme, n_resamples=2000, random_state=7)["as_missing"]
intervals["jackknife"] = differ.coder_jackknife()["as_missing"]

pd.DataFrame(
    {name: {"low": iv.low, "high": iv.high, "width": iv.high - iv.low} for name, iv in intervals.items()}
).T

Now repeat the study the way the broad claim imagines it: **new panels of 10 coders from the same pool, each rating 200 new units**. Where do their alphas fall?

In [ ]:
replications = []
for rep in range(40):
    new_panel = draw_coders(10, **POOL, seed=500 + rep)
    new_sample, _ = rate(new_panel, seed=900 + rep)
    replications.append(make_study(new_sample).alpha().alpha)
replications = np.array(replications)

for name in ("units", "two_way"):
    iv = intervals[name]
    inside = ((replications >= iv.low) & (replications <= iv.high)).mean()
    print(f"{name:<8} interval [{iv.low:.3f}, {iv.high:.3f}] holds {inside:.0%} of the new studies' alphas")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.plot(replications, np.zeros_like(replications) + np.random.default_rng(0).uniform(-0.15, 0.15, len(replications)),
        "o", color="gray", alpha=0.6, label="alpha of a new panel on new units")
for y, name, color in ((0.6, "units", "tab:blue"), (0.9, "two_way", "tab:orange")):
    iv = intervals[name]
    ax.hlines(y, iv.low, iv.high, color=color, linewidth=4, label=f"{name} interval")
ax.axvline(differ.alpha, color="tab:red", linestyle="--", label="this study's alpha")
ax.set_yticks([])
ax.set_ylim(-0.4, 1.3)
ax.set_xlabel("alpha")
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

The units interval is narrow and honest **about these coders**, but most new panels land outside it: a different panel has a different alpha. The two-way interval is wider because it also resamples the coders, and it holds most of them.

The two-way scheme can only be as good as the panel it starts from: with 10 coders, it resamples 10. In the guidance's simulations it covered 92–98% with 10 coders or more when coders differed, and it errs on the wide side when they do not. The `coders` bootstrap was too narrow there (79–92%); for *these units with new coders*, prefer the jackknife.

**Try it:** set `POOL = dict(accuracy=0.8, spread=0.0)` so the coders are alike. The units interval then holds the new studies too, and two-way is wider than it needs to be. This is the case of repeated runs of one LLM judge configuration: runs differ only by chance, so the units interval already covers new runs.

## 7. Cheat sheet

| You want to say | Call |
| --- | --- |
| how reliable **these coders** are on units like these | `result.bootstrap(scheme="units", method="bca")`, falling back to `method="percentile"` when BCa fails |
| ... when units come in groups | declare `group=` in `RatingFrame.from_long`; `scheme="units"` then resamples groups |
| how reliable **coders like these** are on new units | `result.bootstrap(scheme="two_way")`, with 10 coders or more |
| how much alpha depends on **which coders** rated these units | `result.coder_jackknife()` |
| whether one configuration is more reliable than another, on the same units | `study.compare(a=..., b=..., n_resamples=...)`: a paired interval on the difference |
| the published KALPHA interval | `result.bootstrap(scheme="kalpha")`; not for new claims |

Always report the scheme and the number of resamples with the interval; `boot.summary()` and `boot.to_json()` include both, with the seed.

For the full simulation results behind these recommendations, see the [bootstrap coverage guidance](https://github.com/Sathapana/reliability-lab/blob/main/docs/guidance/bootstrap-coverage.md).